# 03 — Inference Demo
Run the full Arabic OCR pipeline on sample images and visualize JSON output

In [ ]:
import sys, json
sys.path.insert(0, '..')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from pathlib import Path
from src.models.detector import FieldDetector
from src.models.ocr_engine import create_ocr_engine
from src.models.pipeline import ArabicIDOCRPipeline
from src.data.preprocess import load_image

## Load Pipeline

In [ ]:
MODEL_PATH = '../runs/train/arabic_id_detector/weights/best.pt'
# Falls back to pretrained YOLO base model if training hasn't run yet
import os
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = 'yolo11n.pt'
    print('WARNING: Using base YOLO model — run training first for accurate field detection')

detector = FieldDetector(model_path=MODEL_PATH, conf_threshold=0.25)
ocr = create_ocr_engine(engine='easyocr', languages=['ar', 'en'], gpu=False)
pipeline = ArabicIDOCRPipeline(detector=detector, ocr_engine=ocr)
print('Pipeline loaded')

## Run Inference on Test Images

In [ ]:
test_images = list(Path('../Egyptain-Person-ID-1/test/images').glob('*.jpg'))[:3]

for img_path in test_images:
    result = pipeline.process_file(str(img_path))
    image = load_image(str(img_path))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # Left: image with bounding boxes
    ax1.imshow(image)
    colors = plt.cm.tab20.colors
    for i, det in enumerate(result.get('raw_detections', [])):
        x1, y1, x2, y2 = det['box_xyxy']
        color = colors[i % len(colors)]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor=color, facecolor='none')
        ax1.add_patch(rect)
        label = f"{det['class_name']}: {det.get('ocr_text', '')[:15]}"
        ax1.text(x1, y1 - 5, label, color=color, fontsize=7, fontweight='bold')
    ax1.set_title(f'{img_path.name} | Side: {result["id_side"]}')
    ax1.axis('off')
    
    # Right: JSON output
    json_text = json.dumps(result['fields'], ensure_ascii=False, indent=2)
    ax2.text(0.05, 0.95, json_text, transform=ax2.transAxes, fontsize=9,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax2.set_title('Extracted Fields (JSON)')
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    print(f'Processing time: {result["metadata"]["processing_time_s"]}s')
    print()